# Clase 5 — Entradas validadas con Pydantic

**Módulo:** Fundamentos

**Intención:** definir con precisión los datos que una API de predicción acepta antes de intentar calcular una predicción.

## Antes de comenzar

Actualiza tu copia del repositorio del curso desde su raíz y reconstruye el ambiente publicado por el profesor:

```bash
git status
git pull
uv sync --locked
```

`uv sync --locked` instala exactamente las versiones registradas en `uv.lock`. En este repositorio no ejecutes `uv add` ni edites `pyproject.toml` o `uv.lock`: la clase usa el ambiente publicado.

**Prerrequisitos:** una aplicación FastAPI con rutas `GET`, parámetros de path y query, y experiencia probando desde `/docs`, Postman y `curl`.

**Resultado observable:** `POST /predicciones` recibe un JSON, lo valida con Pydantic y devuelve una respuesta documentada; una entrada que incumple el esquema recibe `422` y una solicitud válida durante la ventana de mantenimiento de la demostración recibe `503`.

## 1. Del `GET` de la Clase 4 a una solicitud estructurada

En la Clase 4 consultamos un viaje con parámetros simples:

```text
GET /viajes/42?pasajeros=2
```

La URL tiene una parte que identifica el recurso (`/viajes/42`) y otra que agrega opciones pequeñas (`?pasajeros=2`). Este formato funciona bien para consultar algo ya identificado.

Una predicción necesita varios datos que pertenecen al mismo viaje. Enviarlos como muchos parámetros en la URL sería difícil de leer, validar y ampliar. Por eso el cliente enviará un **cuerpo JSON** en una solicitud `POST`.

En una solicitud HTTP, cada parte cumple una función: el método (`POST`) indica la acción, la ruta (`/predicciones`) identifica dónde se solicita y el cuerpo JSON contiene los datos necesarios para realizarla.

## 2. ¿Por qué `POST /predicciones`?

`POST` permite enviar un objeto JSON completo al servidor:

```json
{
  "distancia_km": 4.2,
  "pasajeros": 2,
  "hora_recoleccion": 18
}
```

Un objeto JSON agrupa pares `nombre: valor`. En este ejemplo, `distancia_km`, `pasajeros` y `hora_recoleccion` son los nombres de los datos; `4.2`, `2` y `18` son sus valores. Las llaves `{}` indican que todo pertenece a una misma solicitud.

Aquí `POST` no significa guardar un viaje. Significa solicitar una operación que usa varios datos de entrada y produce un resultado nuevo: una estimación. El servidor recibirá una solicitud como esta y responderá con otro JSON. Por ejemplo, más adelante podría responder `{"duracion_estimada_minutos": 12.5}`.

Los cuerpos de `GET` no se usan para este caso porque navegadores, herramientas y servicios intermedios no los manejan de manera interoperable. Antes de enviar datos desde una API, necesitamos acordar cuáles son obligatorios, qué tipo tienen y qué valores son aceptables. Ese acuerdo será nuestro **contrato**.

## 3. Recopilación: qué hemos estimado hasta ahora

El problema conductor no comienza hoy. Desde la primera clase hemos usado reglas pequeñas para estimar la duración de un viaje y, en cada paso, hemos añadido una responsabilidad:

| Momento | Código principal | Qué incorporamos |
|---|---|---|
| Clase 1 | `estimar_duracion(distancia_km)` | una regla base de 4 minutos por kilómetro más 2 minutos fijos, y un error para distancias no positivas |
| Clase 2 | `estimar_duracion(distancia_km, pasajeros, fin_de_semana)` | pasajeros, fin de semana y validación antes del cálculo |
| Clase 2 | `resumir_viajes(viajes)` | aplicar la estimación a varios diccionarios sin modificar los originales |
| Tarea 2 | endpoints `GET` de viajes y duración | reutilizar las funciones desde una API local, sin copiar las fórmulas dentro de los endpoints |

Esas funciones son **reglas escritas por nosotros**: nadie las entrenó y no aprendieron patrones desde datos. Han servido como un primer predictor transparente. Más adelante, un modelo de aprendizaje automático ocupará ese lugar, pero conservará la misma pregunta: **¿cuántos minutos durará este viaje?**

La cantidad que queremos estimar se llama **variable objetivo**: `duracion_minutos`. Los datos que usamos para producir la estimación se llaman **features** o variables de entrada. Hasta ahora hemos trabajado o considerado estas:

| Dato | Papel en el recorrido |
|---|---|
| `distancia_km` | feature usada desde la Clase 1 y seleccionada para el primer baseline |
| `pasajeros` | feature añadida en la Clase 2 y conservada en el primer baseline |
| `fin_de_semana` | regla usada en la Clase 2 y la Tarea 2; no forma parte del primer baseline de ML |
| `hora_recoleccion` | nueva feature seleccionada para representar el momento del viaje en el primer baseline |
| origen, destino, día, tráfico o clima | posibles features futuras; todavía no forman parte del contrato |

> En esta clase fijaremos sólo tres features: `distancia_km`, `pasajeros` y `hora_recoleccion`. Mantener una lista pequeña y explícita evita que la API prometa datos que el predictor de la siguiente clase no utilizará.

## 4. Una clase puede describir un tipo de dato

Antes de validar datos conviene separar cuatro ideas que suelen confundirse:

- Una **clase** es una plantilla o descripción que define cómo será un tipo de objeto. Aquí `Viaje` describe los datos que esperamos de un viaje.
- Un **objeto** es un valor que vive en el programa y reúne datos o comportamiento.
- Una **instancia** es un objeto creado a partir de una clase. `viaje` será una instancia de `Viaje`.
- Un **atributo** es un dato nombrado que pertenece a un objeto. En `viaje.pasajeros`, `pasajeros` es el atributo y el punto significa «accede a un dato de este objeto».

La clase también puede declarar tipos esperados. En `distancia_km: float`, `float` comunica que esperamos un número decimal; `int` comunica un número entero. Ejecuta la celda: primero creamos una instancia, después le asignamos atributos y finalmente consultamos uno con la notación de punto.

In [1]:
class Viaje:
    distancia_km: float
    pasajeros: int
    hora_recoleccion: int

viaje = Viaje()  # Una instancia: un objeto concreto creado desde la clase.
viaje.distancia_km = 4.2
viaje.pasajeros = 2
viaje.hora_recoleccion = 18

print(f"Viaje de {viaje.distancia_km} km con {viaje.pasajeros} pasajeros")

# La clase declara tipos esperados, pero Python todavía no los verifica.
viaje.distancia_km = "mucho"

print(Viaje.__annotations__)
print(viaje.distancia_km)

Viaje de 4.2 km con 2 pasajeros
{'distancia_km': <class 'float'>, 'pasajeros': <class 'int'>, 'hora_recoleccion': <class 'int'>}
mucho


## 5. Pydantic: el contrato se vuelve código

Un **contrato** es el acuerdo explícito entre quien envía datos y quien los recibe. Para nuestro viaje, el contrato responderá preguntas como: ¿qué campos son obligatorios?, ¿`pasajeros` debe ser texto o número?, ¿puede existir una distancia negativa?

Un **esquema** es la representación precisa de ese contrato: la lista de campos, sus tipos y, después, sus reglas. Así, el contrato es la idea compartida; el esquema es la forma concreta de expresarla en código y documentación.

[Pydantic](https://docs.pydantic.dev/latest/concepts/models/) permite definir ese esquema con una clase de datos. Una clase que hereda de `BaseModel`, como `SolicitudBasica`, se llama **modelo Pydantic**. **Heredar** significa partir del comportamiento de otra clase: al escribir `class SolicitudBasica(BaseModel)`, nuestra clase recibe las capacidades de validación y conversión de `BaseModel`. Sus instancias siguen siendo objetos: podemos acceder a `solicitud.distancia_km`, no necesitamos operar con un diccionario sin forma.

> **No confundas los dos significados de modelo.** Un modelo Pydantic describe la forma de los datos y decide si una solicitud cumple el contrato. No aprende de ejemplos ni calcula predicciones. Un **modelo de aprendizaje automático** se entrena con datos para aprender una relación —por ejemplo, estimar `duracion_minutos` a partir de distancia, pasajeros y hora—. En la Clase 5 usamos el primero para proteger la entrada; en la Clase 6 conectaremos esa entrada validada con el segundo.

Ejecuta ahora la celda: Pydantic construye una instancia de `SolicitudBasica` y convierte el texto `"4.2"` al número `4.2`. Observa el tipo que imprime al final.

In [2]:
from pydantic import BaseModel


class SolicitudBasica(BaseModel):
    distancia_km: float
    pasajeros: int
    hora_recoleccion: int


solicitud = SolicitudBasica(
    distancia_km="4.2",
    pasajeros=2,
    hora_recoleccion=18,
)

print(solicitud)
print(type(solicitud.distancia_km))

distancia_km=4.2 pasajeros=2 hora_recoleccion=18
<class 'float'>


### Del modelo Pydantic a la API

Ya comprobaste la parte central: el modelo Pydantic puede construir y revisar un objeto incluso fuera de una aplicación web. Ahora FastAPI aprovechará ese mismo esquema. Cuando una función de endpoint recibe un parámetro anotado con una clase que hereda de `BaseModel`, FastAPI:

1. lee el cuerpo de la solicitud como JSON;
2. crea y valida la instancia Pydantic;
3. devuelve `422` si no puede construir una instancia válida;
4. sólo entonces ejecuta la función;
5. incorpora el esquema a OpenAPI y a `/docs`.

Por eso Pydantic es útil aquí: una misma clase define qué recibe la API, protege la lógica que vendrá después y documenta el acuerdo para quien consume el servicio.

## 6. `Field`: reglas y documentación para cada dato

[`Field`](https://docs.pydantic.dev/latest/concepts/fields/) permite añadir reglas y documentación a un atributo del modelo. Cada opción debe responder una pregunta del contrato: ¿qué valores se aceptan?, ¿cómo se explica el dato? o ¿qué ocurre si falta?

| Configuración | Qué expresa |
|---|---|
| `gt`, `ge`, `lt`, `le` | límites numéricos: `gt` y `lt` excluyen el límite; `ge` y `le` lo incluyen |
| `description` | significado y unidad del campo en el esquema y en `/docs` |
| `examples` | valores orientativos que aparecerán en la documentación |
| `default` | valor que se utiliza cuando el campo no se envía |

Las abreviaturas vienen del inglés: *greater than* (`gt`, mayor que), *greater than or equal* (`ge`, mayor o igual), *less than* (`lt`, menor que) y *less than or equal* (`le`, menor o igual). Por ejemplo, `Field(gt=0)` rechaza `0`, mientras que `Field(ge=0)` lo acepta.

### 6.1 Un campo que puede omitirse

Supongamos que recibimos las features numéricas del viaje, pero la zona de origen no siempre está disponible. ¿Deberíamos rechazar toda la solicitud por ese dato faltante? Si la zona no es necesaria para el cálculo, el contrato puede permitir tres casos válidos:

```json
{"zona_origen": "Midtown"}
{}
{"zona_origen": null}
```

En Python, `str | None` expresa dos tipos permitidos: un texto o `None`. El operador `|` forma una **unión de tipos**. En JSON, `null` se convierte en `None`. Sin embargo, aceptar `None` y permitir que el campo se omita son decisiones distintas:

- `zona_origen: str | None` acepta texto o `None`, pero exige que el campo aparezca;
- `zona_origen: str | None = None` también permite omitirlo, porque `None` es el valor predeterminado.

Ejecuta el ejemplo aislado antes de combinar esta idea con las demás reglas. Los tres primeros objetos son válidos; el último muestra qué ocurre cuando un campo acepta `None` pero no tiene valor predeterminado.

In [3]:
from pydantic import ValidationError


class ZonaOpcional(BaseModel):
    zona_origen: str | None = None


print("con texto:", ZonaOpcional(zona_origen="Midtown").zona_origen)
print("omitido:", ZonaOpcional().zona_origen)
print("null explícito:", ZonaOpcional(zona_origen=None).zona_origen)


class ZonaNullableObligatoria(BaseModel):
    zona_origen: str | None


try:
    ZonaNullableObligatoria()
except ValidationError as error:
    print("sin valor predeterminado:", error.errors()[0]["msg"])

con texto: Midtown
omitido: None
null explícito: None
sin valor predeterminado: Field required


### 6.2 Reunir tipos, límites y documentación

Ahora sí reunimos las ideas en un modelo de demostración. `SolicitudConReglas` valida las tres features seleccionadas y agrega temporalmente `zona_origen` para mostrar un campo opcional dentro de un objeto más completo.

La celda construye primero una solicitud válida y usa `model_dump()` para convertir la instancia Pydantic en un diccionario de Python. Después intenta crear otra con distancia negativa. El bloque `try` ejecuta ese intento y `except ValidationError` captura el error esperado para que podamos inspeccionarlo sin detener el notebook. `ValidationError` reúne el campo, la regla incumplida y el valor recibido.

`zona_origen` sirve aquí para practicar la sintaxis de un campo opcional; no se añadirá a `SolicitudPrediccion`, porque no forma parte de las tres features que consumirá el primer baseline. FastAPI utilizará las demás reglas y metadatos para validar el cuerpo JSON y generar su esquema OpenAPI.

In [4]:
from pydantic import Field, ValidationError


class SolicitudConReglas(BaseModel):
    distancia_km: float = Field(gt=0, description="Distancia en kilómetros.")
    pasajeros: int = Field(ge=1, le=6, description="Número de personas.")
    hora_recoleccion: int = Field(ge=0, le=23, examples=[18])
    zona_origen: str | None = Field(
        default=None,
        description="Zona de origen, cuando se conoce.",
        examples=["Midtown"],
    )


valida = SolicitudConReglas(
    distancia_km=4.2,
    pasajeros=2,
    hora_recoleccion=18,
)
print(valida.model_dump())

try:
    SolicitudConReglas(distancia_km=-1, pasajeros=2, hora_recoleccion=18)
except ValidationError as error:
    print(error.errors()[0]["msg"])

{'distancia_km': 4.2, 'pasajeros': 2, 'hora_recoleccion': 18, 'zona_origen': None}
Input should be greater than 0


## 7. Preparar el trabajo local y crear `main.py` desde cero

Esta práctica se construye desde una carpeta vacía para hacer visibles las piezas de FastAPI y Pydantic.

Desde la raíz del repositorio público del curso, crea la carpeta ignorada y entra en ella:

```bash
mkdir -p labs/trabajo-local/clase-05
cd labs/trabajo-local/clase-05
pwd
touch main.py
```

`mkdir -p` crea la ruta si todavía no existe; `cd` cambia la terminal a esa carpeta; `pwd` confirma dónde estás y `touch` crea el archivo vacío. La salida de `pwd` debe terminar en `labs/trabajo-local/clase-05`.

Abre `main.py` desde el explorador de VS Code. Si el comando `code` está disponible, también puedes usar `code main.py`. Escribe el primer estado completo:

```python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="API de predicción de viajes")


class SolicitudPrediccion(BaseModel):
    distancia_km: float
    pasajeros: int
    hora_recoleccion: int


@app.get("/")
def inicio() -> dict[str, str]:
    return {"mensaje": "API activa"}
```

Guarda el archivo. Desde `labs/trabajo-local/clase-05`, inicia el servidor reutilizando el proyecto `uv` de la raíz:

```bash
uv run fastapi dev
```

`uv` encuentra el proyecto padre; no crees otro `pyproject.toml`, `uv.lock` o `.venv` dentro del laboratorio. Abre `http://127.0.0.1:8000/` y luego `/docs`. La ruta `GET /` debe responder, pero todavía no aparecerá un cuerpo JSON de predicción porque `SolicitudPrediccion` aún no está conectada a un endpoint.

**Checkpoint:** antes de continuar, confirma la ruta con `pwd`, conserva el servidor activo en una terminal y comprueba que `git status --short` desde la raíz no incluye `labs/trabajo-local/clase-05/`.

## 8. Añadir restricciones y metadatos con `Field`

Ahora modificamos el mismo `main.py`. Primero reemplaza el import de Pydantic para incluir `Field`:

```python
from pydantic import BaseModel, Field
```

Después reemplaza la primera versión de `SolicitudPrediccion` por esta definición. Cada línea añade una regla observable o ayuda para la documentación:

```python
class SolicitudPrediccion(BaseModel):
    distancia_km: float = Field(
        gt=0,
        description="Distancia estimada del viaje en kilómetros.",
        examples=[4.2],
    )
    pasajeros: int = Field(
        ge=1,
        le=6,
        description="Número de personas que viajan.",
        examples=[2],
    )
    hora_recoleccion: int = Field(
        ge=0,
        le=23,
        description="Hora local de inicio, en formato de 24 horas.",
        examples=[18],
    )
```

Guarda el archivo. El servidor de desarrollo detectará el cambio y recargará la aplicación. Las reglas todavía no se aplican a un request porque falta conectar la clase al endpoint; ese será el siguiente cambio.

**Checkpoint:** revisa los logs del servidor y confirma que la recarga terminó sin errores de importación o sintaxis.

## 9. Registrar `POST /predicciones`

FastAPI reconoce `solicitud: SolicitudPrediccion` como cuerpo JSON. Añade esta función al final de `main.py`:

```python
@app.post("/predicciones")
def solicitar_prediccion(
    solicitud: SolicitudPrediccion,
) -> dict[str, object]:
    return {
        "mensaje": "Solicitud válida",
        "datos_recibidos": solicitud.model_dump(),
    }
```

Guarda y abre `/docs`. Deberías ver `POST /predicciones`, sus tres campos, descripciones, ejemplos y límites. Esa documentación aparece porque FastAPI usa el esquema que Pydantic genera desde la clase.

Selecciona **Try it out**, conserva los ejemplos y ejecuta la solicitud. Debe responder `200` con el mensaje y los datos recibidos.

**Checkpoint:** confirma primero el `200`; no continúes a casos inválidos mientras el recorrido válido no funcione.

## 10. Hacer explícita la respuesta

Una API útil documenta lo que devuelve. Añade `RespuestaValidacion` debajo de `SolicitudPrediccion`:

```python
class RespuestaValidacion(BaseModel):
    mensaje: str
    datos_recibidos: SolicitudPrediccion
```

Después reemplaza el decorador y la función `solicitar_prediccion` anteriores; no conserves dos funciones para la misma ruta:

```python
@app.post("/predicciones", response_model=RespuestaValidacion)
def solicitar_prediccion(
    solicitud: SolicitudPrediccion,
) -> RespuestaValidacion:
    return RespuestaValidacion(
        mensaje="Solicitud válida",
        datos_recibidos=solicitud,
    )
```

`response_model` hace visible el acuerdo de salida en `/docs` y evita que el endpoint devuelva detalles accidentales. Guarda, vuelve a ejecutar la solicitud válida y compara ahora los esquemas de request y response.

## 11. Observar validación automática y `422`

Prueba estos casos en `/docs` antes de usar otra herramienta:

| Solicitud | Resultado esperado |
|---|---|
| los tres campos con valores válidos | `200 OK` y eco de la solicitud |
| falta `hora_recoleccion` | `422 Unprocessable Entity` |
| `pasajeros` es texto | `422 Unprocessable Entity` |
| `distancia_km` es negativa | `422 Unprocessable Entity` |

`422` no es un error misterioso: comunica que FastAPI no pudo construir una instancia válida de `SolicitudPrediccion`. Pydantic protege la función; por eso `solicitar_prediccion` no se ejecuta en estos casos.

## 12. Errores que decide la aplicación: `HTTPException`

Pydantic resuelve errores de forma y rango. Sin embargo, una solicitud puede cumplir el esquema y aun así no poder atenderse por el estado del servicio. Para comunicar esa situación desde un endpoint se usa `HTTPException`.

Primero modifica el import de FastAPI:

```python
from fastapi import FastAPI, HTTPException
```

Después reemplaza nuevamente la función completa para situar la regla antes del `return`:

```python
@app.post("/predicciones", response_model=RespuestaValidacion)
def solicitar_prediccion(
    solicitud: SolicitudPrediccion,
) -> RespuestaValidacion:
    if solicitud.hora_recoleccion == 3:
        raise HTTPException(
            status_code=503,
            detail="Servicio no disponible durante la ventana de mantenimiento.",
        )

    return RespuestaValidacion(
        mensaje="Solicitud válida",
        datos_recibidos=solicitud,
    )
```

Para esta demostración, las 03:00 representan una ventana de mantenimiento. Prueba primero una hora distinta de 3 y después `hora_recoleccion: 3`. Ambas solicitudes cumplen el esquema; la primera devuelve `200` y la segunda `503 Service Unavailable` con el mensaje de `detail`. La diferencia es importante: `422` describe datos que no cumplen el contrato; `503` comunica que el servicio no puede atender temporalmente una solicitud válida.

### 12.1 Probar el contrato desde Postman y `curl`

Después de comprobar `/docs`, repite la solicitud válida con `curl`:

```bash
curl -i -X POST http://127.0.0.1:8000/predicciones \
  -H "Content-Type: application/json" \
  -d '{"distancia_km": 4.2, "pasajeros": 2, "hora_recoleccion": 18}'
```

`-X POST` elige el método. El encabezado `Content-Type: application/json` declara el formato del cuerpo. `-d` contiene los datos que se envían. Repite una solicitud inválida para observar `422` y una solicitud válida con `hora_recoleccion: 3` para observar `503`. El contrato y el estado del servicio deben producir el mismo resultado sin importar si el cliente es `/docs`, Postman o `curl`.

Cuando termines las pruebas, vuelve a la terminal del servidor y presiona `Ctrl+C`. Desde la raíz del repositorio público ejecuta `git status --short`: la carpeta de trabajo local no debe aparecer. No crees una rama ni copies esta práctica a `pcd-entregas-2026`; esta clase no es una entrega.

## 13. Diseñar el contrato del primer modelo de ML

Al inicio recordamos varias funciones escritas a mano para estimar duración. Ahora fijamos el acuerdo que permitirá sustituir esas reglas por un modelo de ML sin cambiar lo que la API espera recibir. Cierra la práctica con esta ficha:

| Pregunta | Decisión inicial |
|---|---|
| Problema | estimar duración de un viaje de taxi |
| Usuario | persona que solicita un viaje |
| Variable objetivo | `duracion_minutos` |
| Features iniciales | `distancia_km`, `pasajeros`, `hora_recoleccion` |
| Salida de la API | `duracion_estimada_minutos` |
| Baseline | artefacto pequeño proporcionado para la siguiente clase |
| Supuestos | sin tráfico en tiempo real, clima ni eventos |

El contrato Pydantic debe reflejar exactamente estas tres features. `fin_de_semana`, origen, destino, tráfico y clima quedan fuera del primer baseline: pueden investigarse después, pero agregarlos hoy produciría una API desalineada con el predictor. En esta clase diseñamos y validamos la entrada; no entrenamos ni optimizamos el modelo de ML.

## 14. Errores frecuentes y diagnóstico

| Síntoma | Causa probable | Siguiente verificación |
|---|---|---|
| `405 Method Not Allowed` | se abrió la ruta con `GET` | elegir `POST` en `/docs` |
| `422` inesperado | nombre, tipo o regla no coincide | leer el detalle del campo indicado |
| `NameError: BaseModel` | falta el import | importar desde `pydantic` |
| `/docs` no muestra el cuerpo | el parámetro no usa el tipo Pydantic | revisar la anotación de `solicitud` |
| `503` | el endpoint declaró una indisponibilidad temporal | leer `detail` y revisar el estado del servicio |

## 🚀 Reto opcional

Mejora los `description` y `examples` de los tres campos de `SolicitudPrediccion` para que una persona ajena al curso pueda enviar una solicitud correcta desde `/docs`. No agregues nuevas features: deben mantenerse alineadas con el baseline de la Clase 6.

Referencias para resolverlo:

- [Pydantic — Fields](https://docs.pydantic.dev/latest/concepts/fields/)
- [FastAPI — Body Fields](https://fastapi.tiangolo.com/tutorial/body-fields/)
- [FastAPI — Request Body](https://fastapi.tiangolo.com/tutorial/body/)

## Recopilación

Comenzamos con funciones escritas a mano que estimaban duración y terminamos con un contrato preparado para recibir las features de un predictor. La pregunta se mantiene —cuántos minutos durará un viaje—, pero ahora la entrada tiene una forma explícita y verificable.

Conviene conservar estas ideas:

- una clase común organiza atributos, pero sus anotaciones no validan por sí solas;
- un modelo Pydantic describe y valida datos; no es un modelo de aprendizaje automático;
- `Field` añade límites y documentación al esquema;
- `str | None = None` permite texto, `null` o que un campo se omita;
- `422` indica que los datos no cumplen el contrato, mientras que `503` puede comunicar que el servicio no está disponible para procesar una solicitud válida;
- `POST /predicciones` recibe `distancia_km`, `pasajeros` y `hora_recoleccion` como un solo cuerpo JSON.

En la siguiente clase, el mismo `SolicitudPrediccion` alimentará un artefacto de modelo de ML y la respuesta dejará de ser temporal para contener `duracion_estimada_minutos`.

## ✅ Check final de la clase

- Puedo distinguir una clase, una instancia y un diccionario.
- Puedo explicar la diferencia entre un modelo Pydantic y un modelo de aprendizaje automático.
- Puedo distinguir la variable objetivo de las features seleccionadas.
- Puedo explicar la diferencia entre aceptar `None` y permitir que un campo se omita.
- Mi `POST /predicciones` documenta y valida un cuerpo JSON.
- Sé interpretar respuestas `422` y `503`.
- Mi ficha define objetivo, features, salida y límites del baseline.

## 📚 Referencias

- [FastAPI — Request Body](https://fastapi.tiangolo.com/tutorial/body/)
- [Pydantic — Models](https://docs.pydantic.dev/latest/concepts/models/)
- [Pydantic — Fields](https://docs.pydantic.dev/latest/concepts/fields/)
- [FastAPI — Handling Errors](https://fastapi.tiangolo.com/tutorial/handling-errors/)